# Analysis and Visualization

This notebook loads completed checkpoints produced by Task.ipynb. It never retrains a model. Use it to create report figures, inspect random images, rank failures, compare pretrained and scratch configurations, and defend conclusions with both quantitative and visual evidence.

Recommended evidence pattern: numerical result, visualization, example image, then architectural explanation.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "analysis":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "viskom_pipeline.py").exists():
    raise RuntimeError("Open this notebook from the Week11-13 project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.viskom_pipeline import (
    YoloPolygonDataset,
    available_completed_runs,
    build_per_image_analysis_cache,
    clear_analysis_model_cache,
    compare_runs_on_image,
    create_domain_shift_table,
    dataset_index_for_filename,
    experiment_matrix,
    load_analysis_bundle,
    match_prediction_instances,
    plot_domain_shift,
    plot_per_class_metrics,
    plot_results_comparison,
    plot_training_curves,
    predict_dataset_item,
    select_cached_image,
    visualize_prediction_analysis,
)

ORIGINAL_DATASET_ROOT = PROJECT_ROOT / "Roboflow_Dataset"
PERSONAL_DATASET_ROOT = PROJECT_ROOT / "My_Dataset"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "full_runs"
RESULTS_CSV = PROJECT_ROOT / "results.csv"
ANALYSIS_ROOT = PROJECT_ROOT / "analysis"
FIGURE_ROOT = ANALYSIS_ROOT / "figures"
PERSONAL_METADATA_CSV = ANALYSIS_ROOT / "personal_image_metadata.csv"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

SCORE_THRESHOLD = 0.25
IOU_THRESHOLD = 0.50
MODEL_CACHE = {}

# Keep False for an unattended Run All. Change to True only when manually
# inspecting one image in sections 7-9.
RUN_INTERACTIVE_PREVIEWS = False

print(f"Project root: {PROJECT_ROOT}")
print(f"Figures: {FIGURE_ROOT}")

## 1. Available experiments

Only runs with saved run_metadata.json files are displayed. You can begin analysis after one full run completes; comparisons become available as more runs finish.

In [ ]:
COMPLETED_RUN_IDS = available_completed_runs(ARTIFACT_ROOT)
RUN_MATRIX = pd.DataFrame(experiment_matrix())
display(RUN_MATRIX[RUN_MATRIX["run_id"].isin(COMPLETED_RUN_IDS)])
print(f"Completed full runs: {len(COMPLETED_RUN_IDS)}/6")

## 2. Build per-image analysis metrics

The Run All default processes every completed model on `personal_test` and saves `per_image_metrics.csv` beside that run's evaluation results. Later selection and failure ranking reuse these caches instead of rerunning the whole dataset.

This stage is resumable: an existing cache is reused unless `OVERWRITE_CACHE=True`.

In [ ]:
# Run All default: build resumable personal-test caches for all completed runs.
CACHE_RUN_IDS = list(COMPLETED_RUN_IDS)
# Options: original_test or personal_test
CACHE_DATASET = "personal_test"
OVERWRITE_CACHE = False

dataset_options = {
    "original_test": (ORIGINAL_DATASET_ROOT, "test", None),
    "personal_test": (PERSONAL_DATASET_ROOT, "test", PERSONAL_METADATA_CSV),
}
if CACHE_DATASET not in dataset_options:
    raise ValueError("CACHE_DATASET must be original_test or personal_test")
cache_root, cache_split, cache_metadata = dataset_options[CACHE_DATASET]
if not (cache_root / "data.yaml").exists():
    print(f"Dataset is not ready: {cache_root}")
elif not CACHE_RUN_IDS:
    print("No cache run selected. Add one completed run ID to CACHE_RUN_IDS.")
else:
    for run_id in CACHE_RUN_IDS:
        if run_id not in COMPLETED_RUN_IDS:
            print(f"Skipping incomplete run: {run_id}")
            continue
        path = build_per_image_analysis_cache(
            run_id=run_id,
            artifact_root=ARTIFACT_ROOT,
            evaluation_root=cache_root,
            evaluation_split=cache_split,
            evaluation_name=CACHE_DATASET,
            score_threshold=SCORE_THRESHOLD,
            iou_threshold=IOU_THRESHOLD,
            metadata_csv=cache_metadata,
            overwrite=OVERWRITE_CACHE,
        )
        print("Saved:", path)

## 3. Overall quantitative comparison

These plots use the consolidated results.csv. They become more complete as full evaluations finish.

In [ ]:
if RESULTS_CSV.exists() and len(pd.read_csv(RESULTS_CSV)):
    plot_results_comparison(
        RESULTS_CSV,
        evaluation_dataset="original_test",
        save_path=FIGURE_ROOT / "overall_original_test.png",
    )
else:
    print("No full evaluation rows exist yet in results.csv.")

## 4. Training and convergence curves

Use these figures to explain convergence speed, stability, and pretrained-versus-scratch differences.

In [ ]:
if COMPLETED_RUN_IDS:
    plot_training_curves(
        ARTIFACT_ROOT,
        run_ids=COMPLETED_RUN_IDS,
        save_path=FIGURE_ROOT / "training_curves.png",
    )
else:
    print("No completed full runs exist yet.")

## 5. Per-class performance

Select a completed run and evaluation set. The chart helps connect failures to rare, absent, small, or visually difficult classes.

In [ ]:
PER_CLASS_RUN_ID = COMPLETED_RUN_IDS[0] if COMPLETED_RUN_IDS else None
PER_CLASS_DATASET = "original_test"
TOP_N_CLASSES = 20

if PER_CLASS_RUN_ID:
    plot_per_class_metrics(
        ARTIFACT_ROOT,
        PER_CLASS_RUN_ID,
        PER_CLASS_DATASET,
        top_n=TOP_N_CLASSES,
        save_path=FIGURE_ROOT / f"per_class_{PER_CLASS_RUN_ID}_{PER_CLASS_DATASET}.png",
    )
else:
    print("No completed run is available.")

## 6. Select an image for manual inspection

Supported modes: random, most_false_positives, most_false_negatives, lowest_iou, wrong_class, successful, and specific_image.

Set RANDOM_SEED to None for a different random choice each time the cell runs. Optional class and condition filters may be combined. When nothing qualifies, SELECTED_IMAGE becomes None and the notebook prints a clear message.

In [ ]:
ANALYSIS_RUN_ID = COMPLETED_RUN_IDS[0] if COMPLETED_RUN_IDS else None
ANALYSIS_DATASET = "personal_test"
SELECTION_MODE = "random"
CLASS_FILTER = None
CONDITION_FILTER = None
SPECIFIC_FILENAME = None
RANDOM_SEED = None

SELECTED_IMAGE = None
if RUN_INTERACTIVE_PREVIEWS and ANALYSIS_RUN_ID:
    metrics_path = (
        ARTIFACT_ROOT / ANALYSIS_RUN_ID / "evaluation"
        / ANALYSIS_DATASET / "per_image_metrics.csv"
    )
    SELECTED_IMAGE = select_cached_image(
        metrics_path,
        selection_mode=SELECTION_MODE,
        class_filter=CLASS_FILTER,
        condition_filter=CONDITION_FILTER,
        specific_filename=SPECIFIC_FILENAME,
        random_seed=RANDOM_SEED,
    )
if not RUN_INTERACTIVE_PREVIEWS:
    print("Interactive preview selection skipped for unattended Run All.")
elif SELECTED_IMAGE is None:
    print("No qualifying image was found. Build the cache or change the filters.")
else:
    display(pd.DataFrame([SELECTED_IMAGE]).T.rename(columns={0: "value"}))

## 7. Visualize the selected image

The four panels show the original, ground truth, prediction, and error overlay. Green is a true positive, red is a false positive, yellow is a false negative, and purple is a geometrically matched wrong class.

In [ ]:
if not RUN_INTERACTIVE_PREVIEWS:
    print("Interactive single-image preview skipped.")
elif SELECTED_IMAGE is None:
    print("Nothing to visualize.")
else:
    analysis_options = {
        "original_test": (ORIGINAL_DATASET_ROOT, "test"),
        "personal_test": (PERSONAL_DATASET_ROOT, "test"),
    }
    analysis_root, analysis_split = analysis_options[ANALYSIS_DATASET]
    analysis_dataset = YoloPolygonDataset(analysis_root, analysis_split)
    image_index = dataset_index_for_filename(
        analysis_dataset, SELECTED_IMAGE["filename"]
    )
    if image_index is None:
        print("Selected filename is not present in the dataset.")
    else:
        if ANALYSIS_RUN_ID not in MODEL_CACHE:
            clear_analysis_model_cache(MODEL_CACHE)
        bundle = load_analysis_bundle(
            ANALYSIS_RUN_ID,
            ARTIFACT_ROOT,
            analysis_root,
            model_cache=MODEL_CACHE,
        )
        image, target, prediction = predict_dataset_item(
            bundle,
            analysis_dataset,
            image_index,
            score_threshold=SCORE_THRESHOLD,
        )
        figure_name = (
            f"single_{ANALYSIS_RUN_ID}_{ANALYSIS_DATASET}_"
            f"{Path(SELECTED_IMAGE['filename']).stem}.png"
        )
        figure, match = visualize_prediction_analysis(
            image,
            target,
            prediction,
            analysis_dataset.names,
            run_title=f"{ANALYSIS_RUN_ID} on {SELECTED_IMAGE['filename']}",
            iou_threshold=IOU_THRESHOLD,
            save_path=FIGURE_ROOT / figure_name,
        )
        display(pd.DataFrame([{
            key: value for key, value in match.items()
            if key not in {"iou_matrix", "true_matches", "wrong_class_matches", "unmatched_prediction_indices", "unmatched_ground_truth_indices"}
        }]).T.rename(columns={0: "value"}))

## 8. Pretrained versus scratch on the same image

Choose one model family. This is the most important visual comparison for an NRP ending in 5.

In [ ]:
PAIR_MODEL_KEY = "yolo26"
PAIR_RUNS = {
    "yolo26": ["yolo26n_seg_pretrained", "yolo26n_seg_scratch"],
    "maskrcnn": ["maskrcnn_r50_fpn_v2_pretrained", "maskrcnn_r50_fpn_v2_scratch"],
    "mask2former": ["mask2former_swin_tiny_pretrained", "mask2former_swin_tiny_scratch"],
}
PAIR_FILENAME = SELECTED_IMAGE["filename"] if SELECTED_IMAGE else None

if not RUN_INTERACTIVE_PREVIEWS:
    print("Interactive pretrained-versus-scratch preview skipped.")
elif PAIR_FILENAME is None:
    print("Select an image first.")
elif not all(run_id in COMPLETED_RUN_IDS for run_id in PAIR_RUNS[PAIR_MODEL_KEY]):
    print("Both full runs for this model pair must be completed.")
else:
    pair_root, pair_split = {
        "original_test": (ORIGINAL_DATASET_ROOT, "test"),
        "personal_test": (PERSONAL_DATASET_ROOT, "test"),
    }[ANALYSIS_DATASET]
    compare_runs_on_image(
        PAIR_RUNS[PAIR_MODEL_KEY],
        ARTIFACT_ROOT,
        pair_root,
        pair_split,
        PAIR_FILENAME,
        score_threshold=SCORE_THRESHOLD,
        iou_threshold=IOU_THRESHOLD,
        save_path=FIGURE_ROOT / f"pair_{PAIR_MODEL_KEY}_{Path(PAIR_FILENAME).stem}.png",
    )

## 9. Compare all completed runs on the same image

After all six experiments finish, this creates the strongest presentation view: one image, one ground truth, and all six configurations.

In [ ]:
COMPARE_FILENAME = SELECTED_IMAGE["filename"] if SELECTED_IMAGE else None
COMPARE_RUN_IDS = available_completed_runs(ARTIFACT_ROOT)

if not RUN_INTERACTIVE_PREVIEWS:
    print("Interactive all-run preview skipped.")
elif COMPARE_FILENAME is None:
    print("Select an image first.")
elif not COMPARE_RUN_IDS:
    print("No completed full runs are available.")
else:
    comparison_root, comparison_split = {
        "original_test": (ORIGINAL_DATASET_ROOT, "test"),
        "personal_test": (PERSONAL_DATASET_ROOT, "test"),
    }[ANALYSIS_DATASET]
    compare_runs_on_image(
        COMPARE_RUN_IDS,
        ARTIFACT_ROOT,
        comparison_root,
        comparison_split,
        COMPARE_FILENAME,
        score_threshold=SCORE_THRESHOLD,
        iou_threshold=IOU_THRESHOLD,
        save_path=FIGURE_ROOT / f"all_runs_{Path(COMPARE_FILENAME).stem}.png",
    )

## 10. Domain-shift visualization

This becomes available after all desired runs have been evaluated on both the original and personal test sets.

In [ ]:
if RESULTS_CSV.exists() and len(pd.read_csv(RESULTS_CSV)):
    plot_domain_shift(
        RESULTS_CSV,
        save_path=FIGURE_ROOT / "domain_shift.png",
    )
    DOMAIN_SHIFT_TABLE = create_domain_shift_table(
        RESULTS_CSV,
        save_csv=ANALYSIS_ROOT / "domain_shift_table.csv",
        save_path=FIGURE_ROOT / "domain_shift_table.png",
    )
    if DOMAIN_SHIFT_TABLE is not None:
        display(DOMAIN_SHIFT_TABLE.round(4))
else:
    print("No full evaluation results are available yet.")